In [1]:
import os

import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 100)

NUTRIENTS_PKL = "recipes_nutrients.pkl"
RULES_CSV = "phase_rules.csv"
PARAMS_CSV = "phase_params.csv"
LABEL_QUALITY = ["good", "fair"]       # Recipes eligible for labeling (add "unverified" later if you want)
MAX_PCT_DV = 200                       # Any recipe with a nutrient above this % of the daily value per serving is considered implausible and won't be labeled
MIN_KCAL_ELIGIBLE = 100                 # A serving below this (kcal) = sauce/seasoning/side, not a meal; won't be labeled
MAX_KCAL = 1200                         # A serving above this (kcal) is implausible (usually a whole recipe counted as one serving)
SCORE_MODE = "density"                  # "density" = %DV per 100 kcal (default), "serving" = %DV per serving (old method)
DENSITY_CAP = 20                        # %DV per 100 kcal that gives a nutrient its full score
MIN_KCAL_DENSITY = 100                  # Minimum kcal used in the density calculation (so very light dishes don't get inflated values)
PHASES = ["menstrual", "follicular", "ovulatory", "luteal"]

df = pd.read_pickle(NUTRIENTS_PKL)
print(df.shape)
print(df["quality"].value_counts().to_dict())

(42736, 46)
{'unverified': 17687, 'poor': 10590, 'good': 9782, 'fair': 4677}


In [2]:
# Key = nutrient name in the table (without _ps)  ->  (daily value, unit, display name)
DV = {
    "iron_mg":       (18,   "mg",  "Iron"),
    "vitc_mg":       (90,   "mg",  "Vitamin C"),
    "magnesium_mg":  (420,  "mg",  "Magnesium"),
    "calcium_mg":    (1300, "mg",  "Calcium"),
    "zinc_mg":       (11,   "mg",  "Zinc"),
    "vitb6_mg":      (1.7,  "mg",  "B6"),
    "folate_ug":     (400,  "mcg", "Folate"),
    "vitb12_ug":     (2.4,  "mcg", "B12"),
    "fiber_g":       (28,   "g",   "Fiber"),
    "protein_g":     (50,   "g",   "Protein"),
    "potassium_mg":  (4700, "mg",  "Potassium"),
}
missing_cols = [k for k in DV if f"{k}_ps" not in df.columns]
print("columns missing from recipes_nutrients.pkl:", missing_cols)
DV = {k: v for k, v in DV.items() if k not in missing_cols}

columns missing from recipes_nutrients.pkl: []


In [3]:
TIER_WEIGHT = {"strong": 3, "medium": 2, "weak": 1, "general": 0.5}

DEFAULT_RULES = [
    # phase, nutrient, tier, gate, reason, source
    ("menstrual", "iron_mg", "strong", "Y", "RDA for women aged 19-50 is 18 mg vs. 8 mg for men; iron deficiency is more common with heavy bleeding", "NIH ODS Iron fact sheet"),
    ("menstrual", "vitc_mg", "medium", "Y", "Improves absorption of plant-based (non-heme) iron", "NIH ODS Iron fact sheet"),
    ("menstrual", "magnesium_mg", "weak", "N", "Mixed evidence for relieving menstrual symptoms", "Parazzini 2017; Boyle 2017"),
    ("menstrual", "folate_ug", "general", "N", "General diet quality; no phase-specific evidence", "general"),
    ("menstrual", "vitb12_ug", "general", "N", "General diet quality; no phase-specific evidence", "general"),
    ("menstrual", "protein_g", "general", "N", "General diet quality; no phase-specific evidence", "general"),
    ("follicular", "zinc_mg", "weak", "Y", "Limited evidence for zinc and menstrual symptoms (Brown 2024); no phase-specific evidence", "Brown 2024"),
    ("follicular", "protein_g", "general", "Y", "General diet quality", "general"),
    ("follicular", "fiber_g", "general", "Y", "General diet quality", "general"),
    ("follicular", "folate_ug", "general", "Y", "General diet quality", "general"),
    ("follicular", "vitb12_ug", "general", "N", "General diet quality", "general"),
    ("ovulatory", "zinc_mg", "weak", "Y", "Limited evidence for zinc and menstrual symptoms (Brown 2024); no phase-specific evidence", "Brown 2024"),
    ("ovulatory", "protein_g", "general", "Y", "General diet quality", "general"),
    ("ovulatory", "fiber_g", "general", "Y", "General diet quality", "general"),
    ("ovulatory", "folate_ug", "general", "Y", "General diet quality", "general"),
    ("ovulatory", "vitb12_ug", "general", "N", "General diet quality", "general"),
    ("luteal", "calcium_mg", "medium", "Y", "Calcium is supported for relieving premenstrual symptoms", "Primary-care evidence summaries; Brown 2024"),
    ("luteal", "vitb6_mg", "medium", "Y", "B6 may help with premenstrual symptoms (high supplement doses are risky)", "Primary-care evidence summaries"),
    ("luteal", "magnesium_mg", "weak", "N", "Mixed, low-quality evidence", "Parazzini 2017; Boyle 2017"),
    ("luteal", "zinc_mg", "weak", "N", "Limited evidence", "Brown 2024"),
    ("luteal", "fiber_g", "general", "N", "General diet quality", "general"),
    ("luteal", "potassium_mg", "general", "N", "General diet quality", "general"),
]
DEFAULT_PARAMS = [
    # phase, gate_pct, gate_min_count, top_share
    ("menstrual", 10, 1, 0.25),
    ("follicular", 10, 2, 0.25),
    ("ovulatory", 10, 2, 0.25),
    ("luteal", 10, 1, 0.25),
]

if not os.path.exists(RULES_CSV):
    rules = pd.DataFrame(DEFAULT_RULES, columns=["phase", "nutrient", "tier", "gate", "reason", "source"])
    rules["weight"] = rules["tier"].map(TIER_WEIGHT)
    rules[["phase", "nutrient", "tier", "weight", "gate", "reason", "source"]].to_csv(RULES_CSV, index=False, encoding="utf-8-sig")
if not os.path.exists(PARAMS_CSV):
    pd.DataFrame(DEFAULT_PARAMS, columns=["phase", "gate_pct", "gate_min_count", "top_share"]).to_csv(PARAMS_CSV, index=False, encoding="utf-8-sig")

rules = pd.read_csv(RULES_CSV)
params = pd.read_csv(PARAMS_CSV).set_index("phase")

bad = rules[~rules["nutrient"].isin(DV)]
if len(bad):
    print("Warning: rules referencing unavailable nutrients (ignored):", bad["nutrient"].unique().tolist())
rules = rules[rules["nutrient"].isin(DV)].copy()
rules

,phase,nutrient,tier,weight,gate,reason,source
0,menstrual,iron_mg,strong,3.0,Y,RDA للنساء 19-50 سنة 18 ملغ مقابل 8 للرجال؛ نقص الحديد أشيع مع غزا...,NIH ODS Iron fact sheet
1,menstrual,vitc_mg,medium,2.0,Y,يحسّن امتصاص الحديد النباتي,NIH ODS Iron fact sheet
2,menstrual,magnesium_mg,weak,1.0,N,أدلة متضاربة لتخفيف أعراض الدورة,Parazzini 2017; Boyle 2017
3,menstrual,folate_ug,general,0.5,N,جودة تغذية عامة، لا دليل خاص بالمرحلة,general
4,menstrual,vitb12_ug,general,0.5,N,جودة تغذية عامة، لا دليل خاص بالمرحلة,general
5,menstrual,protein_g,general,0.5,N,جودة تغذية عامة، لا دليل خاص بالمرحلة,general
6,follicular,zinc_mg,weak,1.0,Y,أدلة محدودة للزنك مع أعراض الدورة (Brown 2024)؛ لا دليل خاص بالمرحلة,Brown 2024
7,follicular,protein_g,general,0.5,Y,جودة تغذية عامة,general
8,follicular,fiber_g,general,0.5,Y,جودة تغذية عامة,general
9,follicular,folate_ug,general,0.5,Y,جودة تغذية عامة,general


In [4]:
pdv = pd.DataFrame({k: df[f"{k}_ps"] / DV[k][0] * 100 for k in DV}, index=df.index)

rep = pd.DataFrame({
    "median_%DV": pdv.median().round(1),
    ">=10% DV": (pdv >= 10).mean().mul(100).round(1),
    ">=20% DV": (pdv >= 20).mean().mul(100).round(1),
    "NaN %": pdv.isna().mean().mul(100).round(1),
})
rep.index = [DV[k][2] for k in rep.index]
rep

,median_%DV,>=10% DV,>=20% DV,NaN %
Iron,8.3,43.0,20.0,0.0
Vitamin C,4.0,38.3,27.9,0.0
Magnesium,5.6,30.7,13.7,0.0
Calcium,5.6,31.2,14.6,0.0
Zinc,6.9,37.6,17.3,0.0
B6,7.7,42.3,23.6,0.0
Folate,9.9,49.7,26.8,0.0
B12,5.6,40.0,25.7,0.0
Fiber,5.5,33.2,17.0,0.0
Protein,11.9,55.4,33.0,0.0


In [5]:
quality_ok = df["quality"].isin(LABEL_QUALITY)
max_pdv = pdv.max(axis=1)
plausible = ((max_pdv <= MAX_PCT_DV) & (df["Calories"] <= MAX_KCAL) & (df["Calories"] >= MIN_KCAL_ELIGIBLE)).to_numpy()
print(f"Implausible (nutrient > {MAX_PCT_DV}% DV or kcal > {MAX_KCAL} or < {MIN_KCAL_ELIGIBLE}): {(~plausible).sum():,} of {len(df):,} | among good/fair: {(~plausible & quality_ok.to_numpy()).sum():,} of {int(quality_ok.sum()):,}")

kcal_for_density = df["Calories"].clip(lower=MIN_KCAL_DENSITY)
dens = pdv.mul(100).div(kcal_for_density, axis=0)     # %DV per 100 kcal


def build(mode, verbose=True):
    res, details = {}, {}
    for ph in PHASES:
        r = rules[rules["phase"] == ph]
        if r.empty:
            print(f"!! No rules for phase {ph}")
            continue
        prm = params.loc[ph]
        P = pdv[r["nutrient"].tolist()]                 # %DV per serving (for the gate and for display)
        W = r["weight"].to_numpy(float)

        if mode == "density":
            S, cap = dens[r["nutrient"].tolist()], DENSITY_CAP
        else:
            S, cap = P, 100
        contrib = np.minimum(S.fillna(0).to_numpy(float), cap) / cap * W
        score = contrib.sum(axis=1) / W.sum()

        gate_cols = r.loc[r["gate"] == "Y", "nutrient"].tolist()
        gate_hits = (pdv[gate_cols] >= prm["gate_pct"]).sum(axis=1)      # The gate is always per serving
        gate = (gate_hits >= prm["gate_min_count"]).to_numpy()

        eligible = gate & quality_ok.to_numpy() & plausible
        thr = np.quantile(score[eligible], 1 - prm["top_share"]) if eligible.any() else np.inf
        label = eligible & (score >= thr)

        res[f"score_{ph}"] = score
        res[f"gate_{ph}"] = gate
        res[f"label_{ph}"] = label
        details[ph] = (r, P, contrib)
        if verbose:
            print(f"{ph:11} gate pass: {gate.sum():6,} | eligible: {eligible.sum():6,} | threshold: {thr:.3f} | labeled: {label.sum():6,}")
    return res, details


IRON_FOODS = {"lentil", "spinach", "ground beef", "chickpea", "kale", "tofu", "black bean", "kidney bean", "molasses", "shrimp"}
has_iron_food = df["parts_norm"].apply(lambda parts: any(p in IRON_FOODS for p in parts))
print(f"== SCORE_MODE = {SCORE_MODE} ==")
res, details = build(SCORE_MODE)
OTHER_MODE = "serving" if SCORE_MODE == "density" else "density"
res_alt, _ = build(OTHER_MODE, verbose=False)

out = df[["RecipeId", "Name", "RecipeCategory", "TotalTime_min", "Calories", "servings_final", "servings_source",
          "quality", "n_ingredients", "parts_norm"]].copy()
for k, v in res.items():
    out[k] = v
import html
out["Name"] = out["Name"].apply(html.unescape)   # &amp; -> &
out["max_pct_dv"] = max_pdv.round(0)
out["implausible"] = ~plausible
out["score_mode"] = SCORE_MODE

Implausible (nutrient > 200% DV or kcal > 1200 or < 100): 10,145 of 42,736 | among good/fair: 2,084 of 14,459
== SCORE_MODE = density ==
menstrual   gate pass: 25,556 | eligible:  7,808 | threshold: 0.352 | labeled:  1,952
follicular  gate pass: 22,688 | eligible:  7,462 | threshold: 0.317 | labeled:  1,866
ovulatory   gate pass: 22,688 | eligible:  7,462 | threshold: 0.317 | labeled:  1,866
luteal      gate pass: 21,118 | eligible:  6,116 | threshold: 0.309 | labeled:  1,529


In [6]:
# Label reason: top nutrients that raised the score (≥ 10% DV), for labeled recipes only
def why_strings(ph):
    r, P, contrib = details[ph]
    names = [DV[n][2] for n in r["nutrient"]]
    pv = P.to_numpy(float)
    lab = res[f"label_{ph}"]
    out_s = np.full(len(df), "", dtype=object)
    for i in np.where(lab)[0]:
        order = np.argsort(-contrib[i])
        parts = [f"{names[j]} {pv[i, j]:.0f}%" for j in order[:3] if pv[i, j] >= 10]
        out_s[i] = "; ".join(parts)
    return out_s

for ph in details:
    out[f"why_{ph}"] = why_strings(ph)

lab_cols = [f"label_{ph}" for ph in details]
out["n_phase_labels"] = out[lab_cols].sum(axis=1)
out["phase_labels"] = out[lab_cols].apply(lambda r: ",".join(c.replace("label_", "") for c in lab_cols if r[c]), axis=1)
out[out["n_phase_labels"] > 0][["Name", "quality", "phase_labels", "why_menstrual", "why_luteal"]].head(10)

,Name,quality,phase_labels,why_menstrual,why_luteal
0,Betty Crocker's Southwestern Guacamole Dip,good,luteal,,B6 63%; Fiber 55%; Magnesium 18%
13,Chilies Corn Muffins,good,menstrual,Vitamin C 69%; Iron 10%; Folate 14%,
19,White Chili,good,"menstrual,follicular,ovulatory,luteal",Vitamin C 54%; Iron 32%; Magnesium 35%,Calcium 36%; B6 30%; Magnesium 35%
22,Spaetzle - Basic Recipe - Art Culinaire Issue,fair,"follicular,ovulatory",,
28,Sun-Dried Tomato Pesto,good,"menstrual,luteal",Vitamin C 56%; Iron 26%; Magnesium 24%,B6 81%; Calcium 18%; Magnesium 24%
39,Butter Crepes,good,"follicular,ovulatory",,
41,Corn Fritters,fair,"follicular,ovulatory",,
53,Zucchini Lasagna (Lasagne) - Low Carb,good,"follicular,ovulatory",,
64,Spinach Squares,good,"follicular,ovulatory",,
76,Lemon Herb Butter,good,menstrual,Vitamin C 75%; Iron 11%; Folate 17%,


In [7]:
print("Number of phases each recipe was labeled with:")
print(out["n_phase_labels"].value_counts().sort_index().to_dict())
print(f"\nRecipes with no label: {(out['n_phase_labels'] == 0).sum():,} of {len(out):,}")

ov = pd.DataFrame({a.replace("label_", ""): {b.replace("label_", ""): int((out[a] & out[b]).sum()) for b in lab_cols} for a in lab_cols})
print("\nOverlap between phases (number of shared recipes):")
ov

Number of phases each recipe was labeled with:
{0: 39633, 1: 780, 2: 1172, 3: 515, 4: 636}

Recipes with no label: 39,633 of 42,736

Overlap between phases (number of shared recipes):


,menstrual,follicular,ovulatory,luteal
menstrual,1952,882,882,1093
follicular,882,1866,1866,905
ovulatory,882,1866,1866,905
luteal,1093,905,905,1529


In [8]:
for ph in details:
    sub = out[out[f"label_{ph}"]].sort_values(f"score_{ph}", ascending=False)
    print(f"\n=== {ph}: top 10 ===")
    print(sub[["Name", f"score_{ph}", f"why_{ph}"]].head(10).to_string(index=False))


=== menstrual: top 10 ===
                                           Name  score_menstrual                           why_menstrual
              Spirit's Spinach Stuffed Tomatoes         0.976665 Iron 34%; Vitamin C 115%; Magnesium 32%
                    Chicken , Leeks and Spinach         0.925550  Iron 40%; Vitamin C 67%; Magnesium 40%
                             Ange's Pizza Sauce         0.895577  Iron 29%; Vitamin C 29%; Magnesium 26%
Grilled Mozzarella and Sun-Dried Tomato Skewers         0.888500  Iron 37%; Vitamin C 34%; Magnesium 34%
                          Wildberry Summer Soup         0.865810  Iron 55%; Vitamin C 60%; Magnesium 55%
                   Sesame Carrots and Asparagus         0.864051  Iron 45%; Vitamin C 26%; Magnesium 14%
 Warm Green Bean Salad with Pine Nuts and Basil         0.850628  Iron 22%; Vitamin C 31%; Magnesium 19%
                            Herb-Roasted Salmon         0.849275       Iron 39%; Vitamin C 117%; B12 85%
              Herb Spinach U

In [9]:
# Sanity check: do recipes containing known iron sources have a higher menstrual label rate than the overall average?
IRON_FOODS = {"lentil", "spinach", "ground beef", "chickpea", "kale", "tofu", "black bean", "kidney bean", "molasses", "shrimp"}
has_iron_food = out["parts_norm"].apply(lambda parts: any(p in IRON_FOODS for p in parts))
for ph in ("menstrual",):
    rate_all = out[f"label_{ph}"].mean() * 100
    rate_iron = out.loc[has_iron_food, f"label_{ph}"].mean() * 100
    print(f"{ph}: labeled among ALL = {rate_all:.1f}% | among recipes with iron-rich foods = {rate_iron:.1f}%  (n={int(has_iron_food.sum()):,})")

if "missing_vitc_mg" in df.columns:
    m = out["label_menstrual"].to_numpy()
    print(f"\nLabeled menstrual with at least one ingredient missing vitamin C (not reported in USDA): {(df.loc[m, 'missing_vitc_mg'] > 0).mean()*100:.1f}%")

menstrual: labeled among ALL = 4.6% | among recipes with iron-rich foods = 12.3%  (n=3,725)

Labeled menstrual with at least one ingredient missing vitamin C (not reported in USDA): 8.1%


In [10]:
def summarize(res_, name):
    labs = pd.DataFrame({ph: res_[f"label_{ph}"] for ph in PHASES if f"label_{ph}" in res_})
    n = labs.sum(axis=1)
    iron_rate = labs.loc[has_iron_food.to_numpy(), "menstrual"].mean() * 100
    all_rate = labs["menstrual"].mean() * 100
    return {"mode": name,
            "labeled_any": int((n > 0).sum()),
            "in_3+_phases": int((n >= 3).sum()),
            "in_4_phases": int((n == 4).sum()),
            "menstrual&luteal % of menstrual": round(float((labs["menstrual"] & labs["luteal"]).sum() / max(labs["menstrual"].sum(), 1) * 100), 1),
            "median_kcal_labeled": float(df.loc[n.to_numpy() > 0, "Calories"].median()),
            "median_kcal_all": float(df["Calories"].median()),
            "menstrual rate: iron-foods / all (%)": f"{iron_rate:.1f} / {all_rate:.1f}"}

pd.DataFrame([summarize(res, SCORE_MODE + " (chosen)"), summarize(res_alt, OTHER_MODE)]).set_index("mode").T

mode,density (chosen),serving
labeled_any,3103,2979
in_3+_phases,1151,1301
in_4_phases,636,784
menstrual&luteal % of menstrual,56.0,54.7
median_kcal_labeled,247.9,413.2
median_kcal_all,264.9,264.9
menstrual rate: iron-foods / all (%),12.3 / 4.6,12.6 / 4.6


In [11]:
from collections import Counter

units = pd.read_csv("default_units.csv")[["ingredient", "unit", "g_per_unit"]]
imap = pd.read_csv("ingredient_usda_map.csv")[["ingredient", "fdc_id", "usda_description"]]
per100 = pd.read_pickle("usda_nutrients_per100g.pkl")
NUT_COLS = [k for k in DV if k in per100.columns]
look = (units.merge(imap, on="ingredient")
             .merge(per100.reset_index()[["fdc_id"] + NUT_COLS], on="fdc_id", how="left")
             .drop_duplicates("ingredient").set_index("ingredient"))
rid_index = {rid: i for i, rid in enumerate(df["RecipeId"])}
BIG_QTY = 30


def explain(recipe_id, nutrient="iron_mg", show=True):
    r = df.iloc[rid_index[recipe_id]]
    rows = []
    for ing, q in zip(r["parts_norm"], r["quants_num"]):
        big = q >= BIG_QTY
        g = q if big else q * look.at[ing, "g_per_unit"]
        total = g / 100 * np.nan_to_num(look.at[ing, nutrient])
        rows.append({"ingredient": ing, "qty": q, "unit": "g" if big else look.at[ing, "unit"],
                     "grams": round(float(g), 1), "usda": str(look.at[ing, "usda_description"])[:55],
                     f"{nutrient}_total": round(float(total), 2)})
    t = pd.DataFrame(rows)
    tot = t[f"{nutrient}_total"].sum()
    t["share_%"] = (t[f"{nutrient}_total"] / tot * 100).round(0) if tot > 0 else 0
    if show:
        print(f"{r['Name']} | servings={r['servings_final']:.0f} ({r['servings_source']}) | Calories={r['Calories']:.0f} | total {nutrient}={tot:.1f}")
    return t.sort_values(f"{nutrient}_total", ascending=False)


def culprits(nutrient, limit=400, top=15):
    mask = (pdv[nutrient] > MAX_PCT_DV) & quality_ok
    ids = df.loc[mask, "RecipeId"].head(limit)
    cnt = Counter()
    for rid in ids:
        t = explain(rid, nutrient, show=False)
        cnt[t.iloc[0]["ingredient"]] += 1
    res_ = pd.DataFrame(cnt.most_common(top), columns=["ingredient", "n_recipes_as_top_contributor"])
    res_ = res_.merge(look[["usda_description", "unit", "g_per_unit"]], left_on="ingredient", right_index=True, how="left")
    print(f"{nutrient}: {int(mask.sum()):,} recipes above {MAX_PCT_DV}% DV (good/fair), examined {len(ids)}")
    return res_


culprits("iron_mg")

iron_mg: 21 recipes above 200% DV (good/fair), examined 21


,ingredient,n_recipes_as_top_contributor,usda_description,unit,g_per_unit
0,all purpose flour,12,"Wheat flour, white, all-purpose, enriched, bleached",cup,125.0000
1,cake flour,4,"Wheat flour, white, cake, enriched",cup,137.0000
2,egg,3,"Egg, whole, raw, fresh",cup,243.0000
3,peanut,1,"Peanuts, spanish, raw",cup,146.0000
4,spinach,1,"Spinach, raw",oz,28.3495


In [12]:
culprits("calcium_mg")

calcium_mg: 7 recipes above 200% DV (good/fair), examined 7


,ingredient,n_recipes_as_top_contributor,usda_description,unit,g_per_unit
0,mozzarella cheese,3,"Cheese, mozzarella, nonfat",cup,113.0000
1,cheddar cheese,1,"Cheese, cheddar, sharp, sliced",oz,28.3495
2,gruyere cheese,1,"Cheese, gruyere",cup,132.0000
3,egg,1,"Egg, whole, raw, fresh",cup,243.0000
4,spinach,1,"Spinach, raw",oz,28.3495


In [13]:
culprits("vitc_mg")

vitc_mg: 133 recipes above 200% DV (good/fair), examined 133


,ingredient,n_recipes_as_top_contributor,usda_description,unit,g_per_unit
0,lemon juice,21,"Lemon juice, raw",cup,244.0
1,green bell pepper,16,"Peppers, sweet, green, raw",cup,149.0
2,strawberry,13,"Strawberries, raw",cup,152.0
3,lemon zest,12,"Lemon peel, raw",cup,96.0
4,broccoli,10,"Broccoli, raw",cup,91.0
5,red bell pepper,9,"Peppers, sweet, red, raw",cup,149.0
6,yellow bell pepper,8,"Peppers, sweet, yellow, raw",piece,186.0
7,green chili,8,"Sauce, chili, peppers, hot, immature green, canned",cup,245.0
8,parsley,7,"Parsley, fresh",cup,60.0
9,red pepper flake,5,"Peppers, sweet, red, raw",cup,149.0


In [14]:

bad_ids = df.loc[(pdv["iron_mg"] > MAX_PCT_DV) & quality_ok, "RecipeId"]
if len(bad_ids):
    explain(int(bad_ids.iloc[0]), "iron_mg")

Irish Soda Bread | servings=1 (given) | Calories=3573 | total iron_mg=38.0


In [15]:
out.to_pickle("recipes_labeled.pkl")
print(out.shape)
print(out[lab_cols].sum().to_dict())

(42736, 31)
{'label_menstrual': 1952, 'label_follicular': 1866, 'label_ovulatory': 1866, 'label_luteal': 1529}


In [16]:
labels = [c for c in out.columns if c.startswith("label_")]
print("labeled in any phase:", out[labels].any(axis=1).sum())
print("names with &quot;:", out["Name"].str.contains("&quot;", regex=False).sum())


labeled in any phase: 3103
names with &quot;: 0
